# 08\_training\_router\_with\_images.ipynb

Multimodal router training notebook that:

- Loads **train / val / test** router datasets from Parquet, with **image bytes (`image_png`) and text**.
- Builds a **CLIP-based vision encoder** + **Transformer-based text encoder**.
- Trains a **multimodal router classifier** to pick the best VLM per sample.
- Logs metrics and plots to **Weights & Biases (W&B)**, including:
  - Train / validation loss & accuracy per epoch
  - Final test accuracy and confusion matrix
  - Pareto-style **cost vs utility** scatter plot

> **Note:** This notebook assumes you already ran `07_build_train_datasets.ipynb` and produced the `router_*_trainer.parquet` files with `image_png` and routing labels.


In [ ]:
import os
import io
from dataclasses import dataclass, asdict
from pathlib import Path
from typing import List, Dict, Optional

import numpy as np
import pandas as pd
from PIL import Image

import torch
from torch import nn
from torch.utils.data import Dataset, DataLoader

from transformers import (
    AutoTokenizer,
    AutoModel,
    CLIPImageProcessor,
    CLIPVisionModel,
)

import matplotlib.pyplot as plt
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay

import plotly.express as px

try:
    import wandb
    WANDB_AVAILABLE = True
except ImportError:
    print("wandb not installed; W&B logging will be disabled.")
    WANDB_AVAILABLE = False

print("Python, Torch, Transformers, etc. imported.")

In [ ]:
from dataclasses import dataclass, asdict
from pathlib import Path
from typing import Optional
import torch
import numpy as np

# ---- Config dataclass ----
@dataclass
class RouterConfig:
    # Paths
    project_root: Path = Path.cwd().parent  # adjust if needed
    data_root: Path = Path.cwd().parent.parent.parent / "dataset" / "final_dataset"
    router_subdir: str = "router_lexico"  # where 07_build_train_datasets wrote trainer files
    output_dir: Path = Path.cwd() / "saved_output" /"router_training_outputs_final"
    checkpoint_dir: Path = Path.cwd() / "saved_output" /"router_checkpoints_final"

    # Training history JSON (for plots / debugging)
    training_history_path: Optional[Path] = (
        Path.cwd() / "saved_output" / "router_training_outputs_final" / "router_training_history.json"
    )

    def __post_init__(self):
        self.output_dir.mkdir(parents=True, exist_ok=True)
        self.checkpoint_dir.mkdir(parents=True, exist_ok=True)
        if self.training_history_path is not None:
            self.training_history_path.parent.mkdir(parents=True, exist_ok=True)

    # Trainer parquet filenames (one per split)
    train_file: str = "router_train_trainer.parquet"
    val_file: str   = "router_val_trainer.parquet"
    test_file: str  = "router_test_trainer.parquet"

    # Optional: image_root only used as fallback if image_png bytes missing
    image_root: Path = Path.cwd().parent.parent.parent / "dataset" / "which_vlm_data" / "images"

    # Model + tokenizer
    vision_encoder_name: str = "openai/clip-vit-base-patch32"
    text_encoder_name: str   = "bert-base-uncased"
    text_tokenizer_name: str = "bert-base-uncased"
    d_model: int = 384
    num_layers: int = 4
    num_heads: int = 6
    ffn_dim: int = 1536
    dropout: float = 0.1
    max_text_length: int = 256

    use_image: bool = False
    freeze_vision: bool = True
    freeze_text_encoder: bool = False

    # Training hyperparams
    num_epochs: int = 10
    train_batch_size: int = 64
    eval_batch_size: int = 32
    learning_rate: float = 3e-5
    weight_decay: float = 0.01
    max_grad_norm: float = 1.0
    max_text_length: int = 256

    # Loss configuration
    use_soft_labels: bool = True
    ce_weight: float = 1.0
    kl_weight: float = 0.5
    label_smoothing: float = 0.0

    # Misc
    num_workers: int = 4
    device: str = "cuda" if torch.cuda.is_available() else "cpu"
    seed: int = 42

    # Checkpointing
    best_checkpoint_name: str = "router_best.pt"

    # Logging
    use_wandb: bool = True
    wandb_project: str = "vlm-router"
    wandb_entity: Optional[str] = None  # set if you have a W&B team
    wandb_run_name: str = "router_with_images_final"


config = RouterConfig()

# ---- Seeding ----
def set_seed(seed: int):
    import random
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

set_seed(config.seed)

print(config)
print("Using device:", config.device)


In [ ]:
# # ---- Config dataclass ----
# @dataclass
# class RouterConfig:
#     # Paths
#     project_root: Path = Path.cwd().parent  # adjust if needed
#     data_root: Path = Path.cwd().parent.parent.parent / "dataset" / "final_dataset" 
#     router_subdir: str = "router_lexico"  # where 07_build_train_datasets wrote trainer files
#     output_dir: Path = Path.cwd() / "router_training_outputs_final"
#     checkpoint_dir: Path = Path.cwd() / "router_checkpoints_final"

#     def __post_init__(self):
#         self.output_dir.mkdir(parents=True, exist_ok=True)
#         self.checkpoint_dir.mkdir(parents=True, exist_ok=True)
        
#     # Trainer parquet filenames (one per split)
#     train_file: str = "router_train_trainer.parquet"
#     val_file: str   = "router_val_trainer.parquet"
#     test_file: str  = "router_test_trainer.parquet"

#     # Optional: image_root only used as fallback if image_png bytes missing
#     image_root: Path = Path.cwd().parent.parent.parent / "dataset" / "which_vlm_data" / "images"

#     # Model + tokenizer
#     vision_encoder_name: str = "openai/clip-vit-base-patch32"
#     text_encoder_name: str   = "distilbert-base-uncased"
#     text_tokenizer_name: str = "distilbert-base-uncased"

#     use_image: bool = False
#     freeze_vision: bool = True
#     freeze_text_encoder: bool = False

#     # # Training hyperparams
#     # num_epochs: int = 6          # was 10
#     # train_batch_size: int = 128  # was 64 (drop to 64 if OOM)
#     # eval_batch_size: int = 64    # was 32
#     # learning_rate: float = 2e-5  # was 3e-5
#     # weight_decay: float = 0.01
#     # max_grad_norm: float = 1.0
#     # max_text_length: int = 256

#     # # Loss configuration (for soft labels)
#     # use_soft_labels: bool = True
#     # ce_weight: float = 0.7       # was 1.0
#     # kl_weight: float = 0.3       # was 0.5
#     # label_smoothing: float = 0.05  # was 0.0
    
#     use_image: bool = True
#     freeze_vision: bool = True        # unfreeze later if needed
#     freeze_text_encoder: bool = False

#     # Training hyperparams
#     num_epochs: int = 5
#     train_batch_size: int = 64        # more memory pressure now
#     eval_batch_size: int = 32
#     learning_rate: float = 1e-5       # more conservative with two encoders
#     weight_decay: float = 0.01
#     max_grad_norm: float = 1.0
#     max_text_length: int = 256

#     # Loss configuration
#     use_soft_labels: bool = True
#     ce_weight: float = 0.7
#     kl_weight: float = 0.3
#     label_smoothing: float = 0.05

#     # Misc
#     num_workers: int = 4
#     device: str = "cuda" if torch.cuda.is_available() else "cpu"
#     seed: int = 42

#     # Logging
#     use_wandb: bool = True
#     wandb_project: str = "vlm-router"
#     wandb_entity: Optional[str] = None
#     wandb_run_name: str = "router_text_final"


In [ ]:
# ---- Load trainer parquet datasets ----
router_dir = config.data_root / config.router_subdir
print("Router data dir:", router_dir)

train_path = router_dir / config.train_file
val_path   = router_dir / config.val_file
test_path  = router_dir / config.test_file

print("Train path:", train_path)
print("Val path  :", val_path)
print("Test path :", test_path)

train_df = pd.read_parquet(train_path)
val_df   = pd.read_parquet(val_path)
test_df  = pd.read_parquet(test_path)

print("\nShapes:")
print("train_df:", train_df.shape)
print("val_df  :", val_df.shape)
print("test_df :", test_df.shape)

print("\nTrain columns (first 40):")
print(train_df.columns[:40])

display(train_df.head())

In [ ]:
# ---- Extract utility scheme and hierarchical weights from trainer data ----
if "utility_scheme" in train_df.columns:
    DATA_UTILITY_SCHEME = str(train_df["utility_scheme"].iloc[0])
else:
    DATA_UTILITY_SCHEME = "unknown"

HIER_W_SAMPLE = float(train_df.get("hier_w_sample", pd.Series([np.nan])).iloc[0])
HIER_W_TASK   = float(train_df.get("hier_w_task",   pd.Series([np.nan])).iloc[0])
HIER_W_GLOBAL = float(train_df.get("hier_w_global", pd.Series([np.nan])).iloc[0])

print("Utility metadata from trainer dataset:")
print("  utility_scheme:", DATA_UTILITY_SCHEME)
print("  hier_w_sample :", HIER_W_SAMPLE)
print("  hier_w_task   :", HIER_W_TASK)
print("  hier_w_global :", HIER_W_GLOBAL)

In [ ]:
# ---- Visual sanity check: show a random training sample (image + prompt) ----
import random

rand_idx = random.randint(0, len(train_df) - 1)
row = train_df.iloc[rand_idx]

print("Random sample index:", rand_idx)
print("sample_id:", row.get("sample_id"))
print("router_task:", row.get("router_task"))
print("router_best_model_id:", row.get("router_best_model_id"))
print("router_best_model_name:", row.get("router_best_model_name"))
print("\nPrompt:")
print(row.get("prompt_raw"))

if "image_png" in row and isinstance(row["image_png"], (bytes, bytearray, memoryview)):
    img = Image.open(io.BytesIO(row["image_png"])).convert("RGB")
    plt.figure(figsize=(4, 4))
    plt.imshow(img)
    plt.axis("off")
    plt.title("Random training image")
    plt.show()
else:
    print("No image_png bytes found for this sample.")

In [ ]:
# ---- RouterDataset: loads from image_png bytes when available ----
class RouterDataset(Dataset):
    def __init__(
        self,
        df: pd.DataFrame,
        image_root: Path,
        image_processor: CLIPImageProcessor,
        tokenizer,
        config: RouterConfig,
        model_names: List[str],
    ):
        self.df = df.reset_index(drop=True)
        self.image_root = image_root
        self.image_processor = image_processor
        self.tokenizer = tokenizer
        self.config = config
        self.model_names = model_names

        soft_prefix = "router_soft_p_"
        self.soft_label_cols = [
            f"{soft_prefix}{m}" for m in model_names
            if f"{soft_prefix}{m}" in df.columns
        ]
        self.has_soft_labels = len(self.soft_label_cols) == len(model_names)

    def __len__(self):
        return len(self.df)

    def _load_image(self, row) -> torch.Tensor:
        """Load image, preferring image_png bytes, with fallback to disk."""
        # 1) Try image_png bytes
        if "image_png" in row.index:
            img_bytes = row["image_png"]
            if isinstance(img_bytes, (bytes, bytearray, memoryview)):
                try:
                    pil_img = Image.open(io.BytesIO(img_bytes)).convert("RGB")
                    inputs = self.image_processor(images=pil_img, return_tensors="pt")
                    pixel_values = inputs["pixel_values"].squeeze(0)
                    return pixel_values
                except Exception as e:
                    print(f"[WARN] Failed to decode image_png for sample_id={row.get('sample_id', 'NA')}: {e}")

        # 2) Fallback: disk-based loading via image_path
        img_path = row.get("image_path", None)
        if img_path is None or not isinstance(img_path, str):
            return torch.zeros(3, 224, 224)

        full_path = self.image_root / img_path
        if not full_path.exists():
            return torch.zeros(3, 224, 224)

        inputs = self.image_processor(images=str(full_path), return_tensors="pt")
        pixel_values = inputs["pixel_values"].squeeze(0)
        return pixel_values

    def _build_router_text(self, row) -> str:
        prompt = row.get("prompt_raw", "")

        w = row.get("img_width", None)
        h = row.get("img_height", None)
        ar = row.get("img_aspect_ratio", None)
        len_chars = row.get("txt_prompt_length_chars", None)
        len_words = row.get("txt_prompt_length_words", None)

        meta_parts = []
        if len_words is not None and not np.isnan(len_words):
            meta_parts.append(f"PromptLenWords: {int(len_words)}.")
        if len_chars is not None and not np.isnan(len_chars):
            meta_parts.append(f"PromptLenChars: {int(len_chars)}.")
        if w is not None and h is not None and not np.isnan(w) and not np.isnan(h):
            meta_parts.append(f"ImageWidth: {int(w)}. ImageHeight: {int(h)}.")
        if ar is not None and not np.isnan(ar):
            meta_parts.append(f"ImageAR: {float(ar):.2f}.")

        meta_str = " ".join(meta_parts)
        router_text = f"{meta_str} Question: {prompt}"
        return router_text

    def __getitem__(self, idx):
        row = self.df.iloc[idx]

        # Image
        if self.config.use_image:
            pixel_values = self._load_image(row)
        else:
            pixel_values = torch.zeros(3, 224, 224)
        pixel_values = pixel_values.float()

        # Text
        router_text = self._build_router_text(row)
        encoding = self.tokenizer(
            router_text,
            padding="max_length",
            truncation=True,
            max_length=self.config.max_text_length,
            return_tensors="pt",
        )
        input_ids = encoding["input_ids"].squeeze(0)
        attention_mask = encoding["attention_mask"].squeeze(0)

        # Label (hard)
        label = int(row["router_best_model_id"])

        # Soft labels
        if self.has_soft_labels and self.config.use_soft_labels:
            soft = row[self.soft_label_cols].to_numpy(dtype=np.float32)
            soft_labels = torch.from_numpy(soft)
        else:
            soft_labels = torch.zeros(len(self.model_names), dtype=torch.float32)

        return {
            "pixel_values": pixel_values,
            "input_ids": input_ids,
            "attention_mask": attention_mask,
            "label": label,
            "soft_labels": soft_labels,
            "sample_id": row.get("sample_id"),
        }

In [ ]:
# ---- Loss + metrics helpers (with class weighting) ----
def compute_losses(
    logits: torch.Tensor,
    labels: torch.Tensor,
    soft_labels: torch.Tensor,
    config: RouterConfig,
):
    # --- Hard labels: cross-entropy with optional class weights ---
    # If class_weights_tensor is defined, use it; else standard CE.
    if "class_weights_tensor" in globals() and class_weights_tensor is not None:
        ce_loss_fn = nn.CrossEntropyLoss(
            weight=class_weights_tensor,
            label_smoothing=config.label_smoothing,
        )
    else:
        ce_loss_fn = nn.CrossEntropyLoss(
            label_smoothing=config.label_smoothing,
        )

    ce_loss = ce_loss_fn(logits, labels)

    loss = config.ce_weight * ce_loss
    loss_dict = {"loss": loss, "loss_ce": ce_loss}

    # --- Soft label KL-divergence (router_soft_p_*) ---
    if config.use_soft_labels and soft_labels is not None and soft_labels.numel() > 0:
        eps = 1e-8

        target_p = soft_labels.clamp(min=eps)
        target_p = target_p / target_p.sum(dim=-1, keepdim=True).clamp(min=eps)

        log_probs = torch.log_softmax(logits, dim=-1)
        kl = (target_p * (torch.log(target_p + eps) - log_probs)).sum(dim=-1)
        kl_loss = kl.mean()

        loss = loss + config.kl_weight * kl_loss
        loss_dict["loss"] = loss
        loss_dict["loss_kl"] = kl_loss

    # --- Accuracy ---
    preds = logits.argmax(dim=-1)
    acc = (preds == labels).float().mean()
    loss_dict["acc"] = acc

    return loss_dict


In [ ]:
# ---- Build tokenizer, image processor, datasets, dataloaders ----
# Infer model names from soft label columns (or from label_name column)
soft_prefix = "router_soft_p_"
soft_cols = [c for c in train_df.columns if c.startswith(soft_prefix)]
if len(soft_cols) > 0:
    model_names = [c[len(soft_prefix):] for c in soft_cols]
    print("Model names inferred from soft label columns:", model_names)
else:
    # Fallback: use router_best_model_name mappings
    uniq = train_df[["router_best_model_id", "router_best_model_name"]].drop_duplicates()
    uniq = uniq.sort_values("router_best_model_id")
    model_names = uniq["router_best_model_name"].tolist()
    print("Model names inferred from router_best_model_name:", model_names)
    
num_models = len(model_names)
print("Number of models (classes):", num_models)


In [ ]:

# ---- Class weights for imbalanced training ----
# Count how many times each model is the best in the TRAIN split
class_counts = train_df["router_best_model_id"].value_counts().sort_index()

print("Class counts in train:")
for cid, cnt in class_counts.items():
    print(f"  id={cid:<2}  name={model_names[cid]:<25}  count={cnt}")

# Convert to frequencies
freqs = class_counts / class_counts.sum()

# Inverse-frequency weights (more weight = rarer class)
# You can switch to 1/sqrt(freq) if 1/freq is too aggressive
raw_weights = 1.0 / (freqs)

# Normalize so mean weight = 1 (keeps loss scale reasonable)
raw_weights = raw_weights / raw_weights.mean()

print("\nClass weights (normalized):")
for cid, w in raw_weights.items():
    print(f"  id={cid:<2}  name={model_names[cid]:<25}  weight={w:.3f}")

# Turn into tensor aligned with class indices [0 .. num_models-1]
class_weights_tensor = torch.tensor(
    [raw_weights.get(i, 1.0) for i in range(num_models)],
    dtype=torch.float32,
).to(config.device)



In [ ]:



image_processor = CLIPImageProcessor.from_pretrained(config.vision_encoder_name)
tokenizer = AutoTokenizer.from_pretrained(config.text_tokenizer_name)

train_dataset = RouterDataset(train_df, config.image_root, image_processor, tokenizer, config, model_names)
val_dataset   = RouterDataset(val_df,   config.image_root, image_processor, tokenizer, config, model_names)
test_dataset  = RouterDataset(test_df,  config.image_root, image_processor, tokenizer, config, model_names)


train_loader = DataLoader(
    train_dataset,
    batch_size=config.train_batch_size,
    shuffle=True,
    num_workers=config.num_workers,
    pin_memory=True,
)

val_loader = DataLoader(
    val_dataset,
    batch_size=config.eval_batch_size,
    shuffle=False,
    num_workers=config.num_workers,
    pin_memory=True,
)

test_loader = DataLoader(
    test_dataset,
    batch_size=config.eval_batch_size,
    shuffle=False,
    num_workers=config.num_workers,
    pin_memory=True,
)

print("DataLoaders ready.")

In [ ]:
from torch.utils.data import WeightedRandomSampler

# ---- Sample-level weights: 1 / class_count[label] ----
train_labels = train_df["router_best_model_id"].to_numpy()
sample_weights = np.array(
    [1.0 / class_counts[label] for label in train_labels],
    dtype=np.float32,
)

sampler = WeightedRandomSampler(
    weights=sample_weights,
    num_samples=len(sample_weights),  # one "epoch" ~ same size
    replacement=True,
)

train_loader = DataLoader(
    train_dataset,
    batch_size=config.train_batch_size,
    sampler=sampler,        # <- instead of shuffle=True
    num_workers=config.num_workers,
    pin_memory=True,
)


In [ ]:
# ---- Multimodal Router Model ----
class MultimodalRouterModel(nn.Module):
    def __init__(
        self,
        config: RouterConfig,
        num_models: int,
        model_names: List[str],
    ):
        super().__init__()
        self.config = config
        self.num_models = num_models
        self.model_names = model_names

        # Vision encoder (CLIP)
        self.vision = CLIPVisionModel.from_pretrained(config.vision_encoder_name)
        vision_hidden_size = self.vision.config.hidden_size

        # Text encoder (Transformer)
        self.text_encoder = AutoModel.from_pretrained(config.text_encoder_name)
        text_hidden_size = self.text_encoder.config.hidden_size

        # Projection layers to a shared dim
        hidden_dim = max(vision_hidden_size, text_hidden_size)
        self.vision_proj = nn.Linear(vision_hidden_size, hidden_dim)
        self.text_proj   = nn.Linear(text_hidden_size, hidden_dim)

        # Fusion: simple concatenation + MLP
        fused_dim = hidden_dim * 2
        self.fusion_mlp = nn.Sequential(
            nn.Linear(fused_dim, fused_dim),
            nn.ReLU(),
            nn.Dropout(0.1),
            nn.Linear(fused_dim, hidden_dim),
            nn.ReLU(),
        )

        # Classifier
        self.classifier = nn.Linear(hidden_dim, num_models)

        # Optional freezing
        if config.freeze_vision:
            for p in self.vision.parameters():
                p.requires_grad = False

        if config.freeze_text_encoder:
            for p in self.text_encoder.parameters():
                p.requires_grad = False

    def forward(self, pixel_values, input_ids, attention_mask):
        # Vision backbone
        vision_outputs = self.vision(pixel_values=pixel_values)
        # Use CLS / pooled output (last hidden state of CLS token)
        vision_emb = vision_outputs.pooler_output
        vision_emb = self.vision_proj(vision_emb)

        # Text backbone
        text_outputs = self.text_encoder(
            input_ids=input_ids,
            attention_mask=attention_mask,
        )
        # Use [CLS] token representation or mean pooling
        if hasattr(text_outputs, "pooler_output") and text_outputs.pooler_output is not None:
            text_emb = text_outputs.pooler_output
        else:
            # mean pooling
            last_hidden = text_outputs.last_hidden_state
            mask = attention_mask.unsqueeze(-1).float()
            text_emb = (last_hidden * mask).sum(dim=1) / mask.sum(dim=1).clamp(min=1e-6)
        text_emb = self.text_proj(text_emb)

        # Concatenate and fuse
        fused = torch.cat([vision_emb, text_emb], dim=-1)
        fused = self.fusion_mlp(fused)

        logits = self.classifier(fused)
        return logits

In [ ]:
# ---- Loss + metrics helpers ----
def compute_losses(
    logits: torch.Tensor,
    labels: torch.Tensor,
    soft_labels: torch.Tensor,
    config: RouterConfig,
):
    # Hard labels: cross-entropy with optional label smoothing
    ce_loss_fn = nn.CrossEntropyLoss(label_smoothing=config.label_smoothing)
    ce_loss = ce_loss_fn(logits, labels)

    loss = config.ce_weight * ce_loss
    loss_dict = {"loss": loss, "loss_ce": ce_loss}

    # Soft label KL-divergence (router_soft_p_*)
    if config.use_soft_labels and soft_labels is not None and soft_labels.numel() > 0:
        # Avoid log(0)
        eps = 1e-8
        # Target distribution
        target_p = soft_labels.clamp(min=eps)
        target_p = target_p / target_p.sum(dim=-1, keepdim=True).clamp(min=eps)

        # Router distribution
        log_probs = torch.log_softmax(logits, dim=-1)
        kl = (target_p * (torch.log(target_p + eps) - log_probs)).sum(dim=-1)
        kl_loss = kl.mean()

        loss = loss + config.kl_weight * kl_loss
        loss_dict["loss"] = loss
        loss_dict["loss_kl"] = kl_loss

    # Accuracy
    preds = logits.argmax(dim=-1)
    acc = (preds == labels).float().mean()
    loss_dict["acc"] = acc

    return loss_dict

In [ ]:
# ---- Training & evaluation loops ----
def train_one_epoch(model, loader, optimizer, config, epoch, use_wandb=False):
    model.train()
    total_loss = 0.0
    total_acc = 0.0
    total_batches = 0

    for step, batch in enumerate(loader):
        pixel_values = batch["pixel_values"].to(config.device)
        input_ids = batch["input_ids"].to(config.device)
        attention_mask = batch["attention_mask"].to(config.device)
        labels = batch["label"].to(config.device)
        soft_labels = batch["soft_labels"].to(config.device)

        optimizer.zero_grad()
        logits = model(pixel_values, input_ids, attention_mask)
        losses = compute_losses(logits, labels, soft_labels, config)
        loss = losses["loss"]

        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), config.max_grad_norm)
        optimizer.step()

        total_loss += loss.item()
        total_acc += losses["acc"].item()
        total_batches += 1

        if use_wandb and step % 50 == 0:
            wandb.log({
                "train/step_loss": loss.item(),
                "train/step_acc": losses["acc"].item(),
                "train/epoch": epoch,
                "train/step": step,
            })

    avg_loss = total_loss / max(total_batches, 1)
    avg_acc = total_acc / max(total_batches, 1)
    return avg_loss, avg_acc


@torch.no_grad()
def eval_one_epoch(model, loader, config):
    model.eval()
    total_loss = 0.0
    total_acc = 0.0
    total_batches = 0

    for batch in loader:
        pixel_values = batch["pixel_values"].to(config.device)
        input_ids = batch["input_ids"].to(config.device)
        attention_mask = batch["attention_mask"].to(config.device)
        labels = batch["label"].to(config.device)
        soft_labels = batch["soft_labels"].to(config.device)

        logits = model(pixel_values, input_ids, attention_mask)
        losses = compute_losses(logits, labels, soft_labels, config)

        total_loss += losses["loss"].item()
        total_acc += losses["acc"].item()
        total_batches += 1

    avg_loss = total_loss / max(total_batches, 1)
    avg_acc = total_acc / max(total_batches, 1)
    return avg_loss, avg_acc

In [ ]:
# ---- Initialize model, optimizer, auto-resume, W&B ----

best_ckpt_path = config.checkpoint_dir / config.best_checkpoint_name

# Build model
model = MultimodalRouterModel(config, num_models=num_models, model_names=model_names)
model = model.to(config.device)

# Optimizer (only trainable params)
optimizer = torch.optim.AdamW(
    filter(lambda p: p.requires_grad, model.parameters()),
    lr=config.learning_rate,
    weight_decay=config.weight_decay,
)

# --------- Auto-resume from best checkpoint if it exists ---------
start_epoch = 0
best_val_acc = 0.0  # will get updated if we load a checkpoint

if best_ckpt_path.exists():
    ckpt = torch.load(best_ckpt_path, map_location=config.device)
    model.load_state_dict(ckpt["model_state_dict"])
    print(
        f"✅ Found existing checkpoint at {best_ckpt_path}\n"
        f"   -> epoch={ckpt.get('epoch')}, best_val_acc={ckpt.get('best_val_acc', 0.0):.4f}"
    )

    # Try to restore optimizer as well (so LR, moments, etc. continue smoothly)
    if "optimizer_state_dict" in ckpt:
        try:
            optimizer.load_state_dict(ckpt["optimizer_state_dict"])
            print("✅ Loaded optimizer state from checkpoint.")
        except Exception as e:
            print(f"⚠️ Could not load optimizer state, continuing with fresh optimizer: {e}")

    start_epoch = ckpt.get("epoch", -1) + 1
    best_val_acc = ckpt.get("best_val_acc", 0.0)

    if start_epoch >= config.num_epochs:
        print(
            f"⚠️ start_epoch ({start_epoch}) >= num_epochs ({config.num_epochs}); "
            "no further training epochs will run."
        )
else:
    print(f"No existing checkpoint at {best_ckpt_path}, training from scratch.")

# --------- W&B: safe logger + init ---------
def wandb_log_safe(data: dict):
    """Log to W&B but never crash the notebook if the backend dies."""
    if not config.use_wandb:
        return
    if wandb.run is None or getattr(wandb.run, "_is_finished", False):
        return
    try:
        wandb.log(data)
    except Exception as e:
        print(f"[WARN] W&B logging failed, skipping this log: {e}")
        # Optional: permanently disable W&B for this run
        # config.use_wandb = False

run = None

scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
    optimizer,
    T_max=config.num_epochs,
)


if config.use_wandb and WANDB_AVAILABLE:
    tags = [
        "router",
        "multimodal" if config.use_image else "text_only",
        f"utility:{DATA_UTILITY_SCHEME}",
        f"hier_w_sample:{HIER_W_SAMPLE}",
        f"hier_w_task:{HIER_W_TASK}",
        f"hier_w_global:{HIER_W_GLOBAL}",
    ]
    try:
        run = wandb.init(
            project=config.wandb_project,
            entity=config.wandb_entity,
            name=config.wandb_run_name,
            config={
                **asdict(config),
                "data_utility_scheme": DATA_UTILITY_SCHEME,
                "data_hier_w_sample": HIER_W_SAMPLE,
                "data_hier_w_task": HIER_W_TASK,
                "data_hier_w_global": HIER_W_GLOBAL,
                "model_names": model_names,
            },
            tags=tags,
            # this often fixes BrokenPipe issues on clusters / Jupyter
            settings=wandb.Settings(start_method="thread"),
        )
        wandb.watch(model, log="all", log_freq=100)
    except Exception as e:
        print(f"[WARN] W&B init failed, disabling logging: {e}")
        run = None
        config.use_wandb = False
else:
    config.use_wandb = False  # In case wandb isn't installed

# --------- Training history dict (will be filled in training loop) ---------
history = {
    "train_loss": [],
    "train_acc": [],
    "val_loss": [],
    "val_acc": [],
}


In [ ]:
import json

# Use the same best_ckpt_path and start_epoch / best_val_acc
# defined in the previous cell:
#   best_ckpt_path = config.checkpoint_dir / config.best_checkpoint_name
#   start_epoch, best_val_acc already set there

history_file = getattr(config, "training_history_path", None)

for epoch_idx in range(start_epoch, config.num_epochs):
    # epoch_idx is 0-based; use 1-based for printing / logging if you like
    epoch = epoch_idx + 1
    print(f"Epoch {epoch}/{config.num_epochs}")

    train_loss, train_acc = train_one_epoch(
        model,
        train_loader,
        optimizer,
        config,
        epoch,
        use_wandb=config.use_wandb,
    )
    val_loss, val_acc = eval_one_epoch(model, val_loader, config)

    history["train_loss"].append(train_loss)
    history["train_acc"].append(train_acc)
    history["val_loss"].append(val_loss)
    history["val_acc"].append(val_acc)

    print(f"  Train loss: {train_loss:.4f}, acc: {train_acc:.4f}")
    print(f"  Val   loss: {val_loss:.4f}, acc: {val_acc:.4f}")

    # Safe W&B logging
    wandb_log_safe({
        "train/epoch_loss": train_loss,
        "train/epoch_acc": train_acc,
        "val/epoch_loss": val_loss,
        "val/epoch_acc": val_acc,
        "epoch": epoch,
    })

    # Save JSON history to disk (optional but nice to have)
    if history_file is not None:
        try:
            with open(history_file, "w") as f:
                json.dump(history, f, indent=2)
        except Exception as e:
            print(f"[WARN] Failed to write training history JSON: {e}")

    # ---- Best-checkpoint logic: save to disk, not just in memory ----
    if val_acc > best_val_acc:
        best_val_acc = val_acc
        torch.save(
            {
                "model_state_dict": model.state_dict(),
                "optimizer_state_dict": optimizer.state_dict(),
                "config": asdict(config),
                "epoch": epoch_idx,          # store 0-based epoch index
                "best_val_acc": best_val_acc,
                "model_names": model_names,
            },
            best_ckpt_path,
        )
        print(f"  ✅ New best val acc: {best_val_acc:.4f} (checkpoint saved to {best_ckpt_path})")
    scheduler.step()

print("Training complete.")
print("Best val acc:", best_val_acc)

# ---- Restore best weights from disk so all downstream eval uses the best model ----
if best_ckpt_path.exists():
    ckpt = torch.load(best_ckpt_path, map_location=config.device)
    model.load_state_dict(ckpt["model_state_dict"])
    model.to(config.device)
    model.eval()
    print(
        f"Loaded best validation checkpoint from epoch {ckpt.get('epoch')} "
        f"with val_acc={ckpt.get('best_val_acc', best_val_acc):.4f}"
    )
else:
    print("⚠️ No best checkpoint found on disk; using last-epoch model.")


In [ ]:
# # ==== SAVE FINAL MODEL ====
# final_path = os.path.join(config.checkpoint_dir, "router_final.pt")
# torch.save(model.state_dict(), final_path)
# print(f"[Saved final model] → {final_path}")


In [ ]:
# ---- Plot training curves ----
epochs = range(1, config.num_epochs + 1)

plt.figure(figsize=(10, 4))
plt.subplot(1, 2, 1)
plt.plot(epochs, history["train_loss"], label="Train")
plt.plot(epochs, history["val_loss"], label="Val")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.title("Loss vs Epoch")
plt.legend()

plt.subplot(1, 2, 2)
plt.plot(epochs, history["train_acc"], label="Train")
plt.plot(epochs, history["val_acc"], label="Val")
plt.xlabel("Epoch")
plt.ylabel("Accuracy")
plt.title("Accuracy vs Epoch")
plt.legend()

plt.tight_layout()
plt.show()

In [ ]:
# ---- Evaluate on test set & confusion matrix ----
@torch.no_grad()
def evaluate_on_test(model, loader, config):
    model.eval()
    all_labels = []
    all_preds = []

    for batch in loader:
        pixel_values = batch["pixel_values"].to(config.device)
        input_ids = batch["input_ids"].to(config.device)
        attention_mask = batch["attention_mask"].to(config.device)
        labels = batch["label"].to(config.device)

        logits = model(pixel_values, input_ids, attention_mask)
        preds = logits.argmax(dim=-1)

        all_labels.append(labels.cpu().numpy())
        all_preds.append(preds.cpu().numpy())

    all_labels = np.concatenate(all_labels, axis=0)
    all_preds = np.concatenate(all_preds, axis=0)
    acc = (all_labels == all_preds).mean()
    return acc, all_labels, all_preds

test_acc, test_labels, test_preds = evaluate_on_test(model, test_loader, config)
print(f"Test accuracy: {test_acc:.4f}")

# Confusion matrix
cm = confusion_matrix(test_labels, test_preds, labels=list(range(num_models)))
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=model_names)

fig_cm, ax_cm = plt.subplots(figsize=(8, 8))  # NEW: explicitly get fig
disp.plot(include_values=True, xticks_rotation="vertical", cmap="Blues", ax=ax_cm)
plt.title("Router confusion matrix (test set)")
plt.tight_layout()
plt.show()

if config.use_wandb:
    wandb.log({
        "test/accuracy": test_acc,                          # NEW: log accuracy here
        "test/confusion_matrix": wandb.Image(fig_cm),       # NEW: log confusion matrix fig
    })


In [ ]:
# ---- Collect router predictions on test set (sample-level) ----
@torch.no_grad()
def collect_test_predictions(model, loader, df_test, config):
    model.eval()
    all_preds = []
    all_labels = []
    all_sample_ids = []

    # We assume sequential sampler for DataLoader (default when shuffle=False)
    for batch_idx, batch in enumerate(loader):
        pixel_values = batch["pixel_values"].to(config.device)
        input_ids = batch["input_ids"].to(config.device)
        attention_mask = batch["attention_mask"].to(config.device)
        labels = batch["label"].to(config.device)

        logits = model(pixel_values, input_ids, attention_mask)
        preds = logits.argmax(dim=-1)

        all_preds.append(preds.cpu().numpy())
        all_labels.append(labels.cpu().numpy())

        start_idx = batch_idx * config.eval_batch_size
        end_idx = start_idx + preds.size(0)
        batch_indices = list(range(start_idx, end_idx))
        sample_ids = df_test.iloc[batch_indices]["sample_id"].tolist()
        all_sample_ids.extend(sample_ids)

    all_preds = np.concatenate(all_preds, axis=0)
    all_labels = np.concatenate(all_labels, axis=0)

    results = pd.DataFrame({
        "sample_id": all_sample_ids,
        "router_pred_model_id": all_preds,
        "router_label_id": all_labels,
    })

    if "router_best_model_name" in df_test.columns and "router_best_model_id" in df_test.columns:
        id_to_name = (
            df_test[["router_best_model_id", "router_best_model_name"]]
            .drop_duplicates()
            .set_index("router_best_model_id")["router_best_model_name"]
            .to_dict()
        )
        results["router_pred_model_name"] = results["router_pred_model_id"].map(id_to_name)

    return results

router_test_results = collect_test_predictions(model, test_loader, test_df, config)
print("Router test predictions preview:")
display(router_test_results.head())

# Per-class accuracy diagnostics
all_labels = test_labels   # NEW: reuse from evaluate_on_test
all_preds = test_preds

per_class_acc = []
for cid in range(num_models):
    mask = (all_labels == cid)
    if mask.sum() == 0:
        per_class_acc.append(np.nan)
    else:
        per_class_acc.append((all_preds[mask] == all_labels[mask]).mean())

overall_acc = (all_labels == all_preds).mean()  # NEW: define explicitly
balanced_acc = np.nanmean(per_class_acc)
print(f"Overall acc   : {overall_acc:.4f}")
print(f"Balanced acc  : {balanced_acc:.4f}")

for cid, acc in enumerate(per_class_acc):
    print(f"  {model_names[cid]:<25}  acc_when_best = {acc:.3f}")

if config.use_wandb:
    wandb.log({
        "test/overall_acc": overall_acc,
        "test/balanced_acc": balanced_acc,
        **{f"test/class_acc/{model_names[i]}": per_class_acc[i] for i in range(num_models)},
    })


In [ ]:
# Per-class accuracy diagnostics
per_class_acc = []
for cid in range(num_models):
    mask = (all_labels == cid)
    if mask.sum() == 0:
        per_class_acc.append(np.nan)
    else:
        per_class_acc.append((all_preds[mask] == all_labels[mask]).mean())

balanced_acc = np.nanmean(per_class_acc)
print(f"Overall acc   : {overall_acc:.4f}")
print(f"Balanced acc  : {balanced_acc:.4f}")

for cid, acc in enumerate(per_class_acc):
    print(f"  {model_names[cid]:<25}  acc_when_best = {acc:.3f}")

if config.use_wandb:
    wandb.log({
        "val/overall_acc": overall_acc,
        "val/balanced_acc": balanced_acc,
        **{f"val/class_acc/{model_names[i]}": per_class_acc[i] for i in range(num_models)},
    })


In [ ]:
# ---- Finish W&B run ----
if config.use_wandb and run is not None:
    run.finish()
    print("W&B run finished.")